In [12]:
import pandas as pd
import urllib.request

url_direct = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
urllib.request.urlretrieve(url_direct, "sms_data.tsv")
df = pd.read_csv("sms_data.tsv", sep='\t', header=None, names=['label', 'message'])
y = (df['label'] == 'spam').astype(int)

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")

Dataset loaded successfully!
Dataset shape: (5572, 2)


In [13]:
df.tail(5)

,label,message
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [18]:
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import pandas as pd
algorithms = {
    'Perceptron': Perceptron(random_state=42, max_iter=1000),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Linear SVM': LinearSVC(random_state=42, max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'Neural Network': MLPClassifier(random_state=42, max_iter=500),
}

vectorizers = {
    "Count Vectorizer": CountVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2),lowercase=True, ),
    "TF-IDF Vectorizer": TfidfVectorizer(max_features=3000, stop_words='english',ngram_range=(1, 2),lowercase=True, )
}

best_score = 0
best_model = None
best_vectorizer_name = ""
best_vectorizer_obj = None

X_train_text, X_test_text, y_train, y_test = train_test_split(df['message'], y, test_size=0.2, random_state=42, stratify=y)


for vec_name, vectorizer in vectorizers.items():
    X_train_vec = vectorizer.fit_transform(X_train_text).astype(float)
    X_test_vec = vectorizer.transform(X_test_text).astype(float)
    
    for algo_name, model in algorithms.items():
        model.fit(X_train_vec, y_train)
        preds = model.predict(X_test_vec)
        score = f1_score(y_test, preds)
        if score > best_score:
            best_score = score
            best_model = model
            best_vectorizer_name = vec_name
            best_vectorizer_obj = vectorizer

print(f"Winner: {best_model} using {best_vectorizer_name} with F1-Score: {best_score:.4f}")

joblib.dump(best_model, 'best_spam_model.pkl')
joblib.dump(best_vectorizer_obj, 'best_vectorizer.pkl')

print("Files saved successfully:")
print("best_spam_model.pkl")
print("best_vectorizer.pkl")

Winner: MLPClassifier(max_iter=500, random_state=42) using Count Vectorizer with F1-Score: 0.9324
Files saved successfully:
best_spam_model.pkl
best_vectorizer.pkl


In [34]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

model = joblib.load('best_spam_model.pkl')
vectorizer = joblib.load('best_vectorizer.pkl')

class MessageInput(BaseModel):
    message: str

@app.post("/predict-spam")
def predict_spam(data: MessageInput):
    text_vec = vectorizer.transform([data.message])
    prediction = model.predict(text_vec)[0]
    label = "Spam" if prediction == 1 else "Not Spam"
    
    return {
        "message": data.message,
        "prediction": label,
        "model_name": "Neural Network (MLPClassifier)",
        "f1_score": "93.24%",
        "vectorizer": "Count Vectorizer (ngram 1-2)",
        "max_features": 3000
    }
    # uvicorn server2:app --reload

In [33]:
import joblib
import pandas as pd
loaded_model = joblib.load('best_spam_model.pkl')
loaded_vectorizer = joblib.load('best_vectorizer.pkl')
sample_messages =["Congratulations"]

X_sample_vec = loaded_vectorizer.transform(sample_messages)
predictions = loaded_model.predict(X_sample_vec)

for msg, pred in zip(sample_messages, predictions):
    label = "Spam" if pred == 1 else "Not Spam"
    print(f"Prediction: {label}")

Prediction: Not Spam


In [35]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/algorithms-checkpoint.ipynb
	modified:   __pycache__/server2.cpython-313.pyc
	modified:   algorithms.ipynb
	modified:   server2.py
	modified:   src/App.css
	modified:   src/App.jsx

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
git 